## Classic Environment Preflight

This notebook requires the classic runtime. If this check fails, rebuild with `CLASSIC=1 make notebooks-build`, restart the container, and select kernel **Python 3 (classic-langchain)**.


In [ ]:
import os
import sys

def _classic_fail(reason: str) -> None:
    raise RuntimeError(
        f"Classic runtime preflight failed: {reason}\n"
        "Fix:\n"
        "1. CLASSIC=1 make notebooks-build\n"
        "2. make notebooks-up\n"
        "3. In Jupyter, select kernel: Python 3 (classic-langchain)"
    )

kernel_name = os.getenv("JPY_KERNEL_NAME", "")
prefix = sys.prefix.lower()
if "venv-classic" not in prefix and "classic" not in kernel_name.lower():
    _classic_fail(f"detected sys.prefix={sys.prefix!r}, JPY_KERNEL_NAME={kernel_name!r}")

try:
    import langchain  # noqa: F401
except Exception as exc:
    _classic_fail(f"langchain import failed: {exc}")

print("Classic preflight passed.")


# Classic Notebook
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026/langchain/` for current APIs and recommended techniques.


In [ ]:
%pip install -q langchain
%pip install -q crewai
%pip install -q duckduckgo-search ddgs


In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [ ]:
# Prepare the model configuration for CrewAI
import os

# CrewAI accepts provider-prefixed model names
agent_llm = os.getenv("CREWAI_MODEL", "openai/gpt-4o-mini")


In [ ]:
from crewai import Agent, Task, Crew, Process

# Define your agents with roles and goals
researcher = Agent(
    role="Senior Research Analyst",
    goal="Uncover cutting-edge developments in AI and data science",
    backstory="""You work at a leading tech think tank.
  Your expertise lies in identifying emerging trends.
  You have a knack for dissecting complex data and presenting actionable insights.""",
    verbose=True,
    allow_delegation=False,
    llm=agent_llm
  
)
writer = Agent(
    role="Tech Content Strategist",
    goal="Craft compelling content on tech advancements",
    backstory="""You are a renowned Content Strategist, known for your insightful and engaging articles.
  You transform complex concepts into compelling narratives.""",
    verbose=True,
    allow_delegation=True,
    llm=agent_llm
)

# Create tasks for your agents
task1 = Task(
    description="""Conduct a comprehensive analysis of the latest advancements in AI in 2024.
  Identify key trends, breakthrough technologies, and potential industry impacts.
  Your final answer MUST be a full analysis report""",
    agent=researcher,
    expected_output="A structured analysis report with trends, technologies, and impacts.",
    llm=agent_llm,
)

task2 = Task(
    description="""Using the insights provided, develop an engaging blog
  post that highlights the most significant AI advancements.
  Your post should be informative yet accessible, catering to a tech-savvy audience.
  Make it sound cool, avoid complex words so it doesn't sound like AI.
  Your final answer MUST be the full blog post of at least 4 paragraphs.""",
    agent=writer,
    expected_output="A blog post of at least 4 paragraphs based on the research.",
    llm=agent_llm,
)

# Instantiate your crew with a sequential process
crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    verbose=True,
)

# Get your crew to work!
result = crew.kickoff()

print("######################")
print(result)
